In [1]:

import requests
import xml.etree.ElementTree as ET

url = "https://mapy.geoportal.gov.pl/wss/service/rcn"

params = {
    "service": "WFS",
    "request": "GetCapabilities"
}

response = requests.get(url, params=params, timeout=60)

print("Status API:", response.status_code)
response.raise_for_status()

root = ET.fromstring(response.content)

print("\nDostępne warstwy danych:")

for feature in root.findall(".//{*}FeatureType"):
    name = feature.find("{*}Name")
    if name is not None:
        print(name.text)


Status API: 200

Dostępne warstwy danych:
ms:lokale
ms:budynki
ms:dzialki
ms:powiaty


In [2]:
import requests
import xml.etree.ElementTree as ET

url = "https://mapy.geoportal.gov.pl/wss/service/rcn"

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "DescribeFeatureType",
    "typeNames": "ms:lokale"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

root = ET.fromstring(response.content)

fields = []

for element in root.findall(".//{*}element"):
    name = element.get("name")
    if name and name not in fields:
        fields.append(name)

print("Dostępne pola:")
for field in fields:
    print(field)

Dostępne pola:
lokale
GEOMETRIA
gid
SERWIS_RCN
TERYT
TRAN_PRZESTRZEN_NAZW
TRAN_LOKALNY_ID_IIP
TRAN_WERSJA_ID
TRAN_OZNACZENIE_TRANS
TRAN_RODZAJ_TRANS
TRAN_RODZAJ_RYNKU
TRAN_SPRZEDAJACY
TRAN_KUPUJACY
TRAN_CENA_BRUTTO
TRAN_VAT
DOK_OZNACZENIE
DOK_DATA
DOK_TWORCA
NIER_RODZAJ
NIER_PRAWO
NIER_UDZIAL
NIER_POW_GRUNTU
NIER_OPIS
NIER_CENA_BRUTTO
NIER_VAT
LOK_ID_LOKALU
LOK_FUNKCJA
LOK_LICZBA_IZB
LOK_NR_KOND
LOK_POW_UZYT
LOK_POW_PRZYN
LOK_CENA_BRUTTO
LOK_VAT
LOK_INFO
LOK_ADRES


In [3]:

import requests
import xml.etree.ElementTree as ET
import pandas as pd

url = "https://mapy.geoportal.gov.pl/wss/service/rcn"

# Filtr: tylko Kraków
filter_xml = """
<Filter xmlns="http://www.opengis.net/fes/2.0">
    <PropertyIsEqualTo>
        <ValueReference>TERYT</ValueReference>
        <Literal>1261</Literal>
    </PropertyIsEqualTo>
</Filter>
"""

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": "ms:lokale",
    "count": 10,
    "filter": filter_xml
}

response = requests.get(url, params=params, timeout=120)

print("Status:", response.status_code)
response.raise_for_status()

root = ET.fromstring(response.content)

print("Typ odpowiedzi:", root.tag)
print("Liczba pasujących:", root.get("numberMatched"))
print("Liczba zwróconych:", root.get("numberReturned"))

# Odczyt pierwszych rekordów
members = root.findall(".//{*}member")
rows = []

for member in members:
    if len(member) == 0:
        continue

    feature = member[0]
    row = {}

    for element in feature:
        name = element.tag.split("}")[-1]

        if name == "GEOMETRIA":
            continue

        row[name] = "".join(element.itertext()).strip()

    rows.append(row)

df_rcn = pd.DataFrame(rows)

print("\nPobrano rekordów:", len(df_rcn))

if not df_rcn.empty:
    selected = [
        "LOK_CENA_BRUTTO",
        "LOK_POW_UZYT",
        "LOK_LICZBA_IZB",
        "LOK_NR_KOND",
        "LOK_ADRES",
        "DOK_DATA",
        "TRAN_RODZAJ_RYNKU"
    ]

    display(df_rcn.reindex(columns=selected).head(10))
else:
    print(ET.tostring(root, encoding="unicode")[:2000])


Status: 200
Typ odpowiedzi: {http://www.opengis.net/wfs/2.0}FeatureCollection
Liczba pasujących: unknown
Liczba zwróconych: 0

Pobrano rekordów: 0
<ns0:FeatureCollection xmlns:ns0="http://www.opengis.net/wfs/2.0" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://mapserver.gis.umn.edu/mapserver https://mapy.geoportal.gov.pl/wss/service/rcn?SERVICE=WFS&amp;VERSION=2.0.0&amp;REQUEST=DescribeFeatureType&amp;TYPENAME=ms:lokale&amp;OUTPUTFORMAT=application%2Fgml%2Bxml%3B%20version%3D3.2 http://www.opengis.net/wfs/2.0 http://schemas.opengis.net/wfs/2.0/wfs.xsd http://www.opengis.net/gml/3.2 http://schemas.opengis.net/gml/3.2.1/gml.xsd" timeStamp="2026-11-17T20:57:35" numberMatched="unknown" numberReturned="0">
</ns0:FeatureCollection>


In [5]:


import requests
import xml.etree.ElementTree as ET
import pandas as pd

url = "https://mapy.geoportal.gov.pl/wss/service/rcn"

params = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "REQUEST": "GetFeature",
    "TYPENAMES": "ms:lokale",
    "COUNT": 5
}

try:
    r = requests.get(
        url,
        params=params,
        timeout=(10, 40)
    )

    print("Status:", r.status_code)
    r.raise_for_status()

    root = ET.fromstring(r.content)

    print("Liczba pasujących:",
          root.get("numberMatched"))
    print("Liczba zwróconych:",
          root.get("numberReturned"))

    members = root.findall(".//{*}member")
    rows = []

    for member in members:
        if len(member) == 0:
            continue

        feature = member[0]
        row = {}

        for element in feature:
            name = element.tag.split("}")[-1]

            if name == "GEOMETRIA":
                continue

            row[name] = "".join(
                element.itertext()
            ).strip()

        rows.append(row)

    df_test = pd.DataFrame(rows)

    print("Pobrano rekordów:", len(df_test))

    if not df_test.empty:
        display(df_test[[
            c for c in [
                "TERYT",
                "LOK_CENA_BRUTTO",
                "LOK_POW_UZYT",
                "LOK_ADRES"
            ]
            if c in df_test.columns
        ]])
    else:
        print("Odpowiedź:", r.text[:1000])

except requests.exceptions.RequestException as e:
    print("Błąd połączenia:", e)


Status: 200
Liczba pasujących: unknown
Liczba zwróconych: 5
Pobrano rekordów: 5


,TERYT,LOK_CENA_BRUTTO,LOK_POW_UZYT,LOK_ADRES
0,1465,640000,34.2,MSC:Warszawa;UL:ulica Zawiszy;NR_PORZ:12
1,2061,41000,1080.3,MSC:Białystok;UL:Nowowarszawska;NR_PORZ:132
2,2061,55000,885.58,MSC:Białystok;UL:Świętego Rocha;NR_PORZ:5A
3,2061,400000,50.39,MSC:Białystok;UL:Podleśna;NR_PORZ:17
4,2061,558000,70.51,MSC:Białystok;UL:Aloesowa;NR_PORZ:3S


In [6]:

import requests
import xml.etree.ElementTree as ET

url = "https://mapy.geoportal.gov.pl/wss/service/rcn"

miasta = {
    "Warszawa": "1465",
    "Białystok": "2061",
    "Kraków": "1261"
}

for miasto, teryt in miasta.items():

    filter_xml = f"""
    <fes:Filter xmlns:fes="http://www.opengis.net/fes/2.0">
        <fes:PropertyIsEqualTo>
            <fes:ValueReference>TERYT</fes:ValueReference>
            <fes:Literal>{teryt}</fes:Literal>
        </fes:PropertyIsEqualTo>
    </fes:Filter>
    """

    params = {
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "TYPENAMES": "ms:lokale",
        "COUNT": 5,
        "FILTER": filter_xml
    }

    try:
        r = requests.get(
            url,
            params=params,
            timeout=(10, 40)
        )
        r.raise_for_status()

        root = ET.fromstring(r.content)

        members = root.findall(
            ".//{http://www.opengis.net/wfs/2.0}member"
        )

        returned_codes = []

        for member in members:
            if len(member) == 0:
                continue

            feature = member[0]

            for element in feature:
                if element.tag.split("}")[-1] == "TERYT":
                    returned_codes.append(element.text)

        print(f"\n{miasto}")
        print("Status:", r.status_code)
        print("Liczba rekordów:", len(members))
        print("Zwrócone TERYT:", returned_codes)

    except Exception as e:
        print(f"\n{miasto} – błąd:", e)


Warszawa
Status: 200
Liczba rekordów: 5
Zwrócone TERYT: ['1465', '1465', '1465', '1465', '1465']

Białystok
Status: 200
Liczba rekordów: 5
Zwrócone TERYT: ['2061', '2061', '2061', '2061', '2061']

Kraków
Status: 200
Liczba rekordów: 0
Zwrócone TERYT: []


In [7]:

import requests
import xml.etree.ElementTree as ET

url = "https://mapy.geoportal.gov.pl/wss/service/rcn"

filter_xml = """
<fes:Filter xmlns:fes="http://www.opengis.net/fes/2.0">
    <fes:PropertyIsEqualTo>
        <fes:ValueReference>TERYT</fes:ValueReference>
        <fes:Literal>1465</fes:Literal>
    </fes:PropertyIsEqualTo>
</fes:Filter>
"""

params = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "REQUEST": "GetFeature",
    "TYPENAMES": "ms:lokale",
    "RESULTTYPE": "hits",
    "FILTER": filter_xml
}

r = requests.get(url, params=params, timeout=120)
r.raise_for_status()

root = ET.fromstring(r.content)

print("Status:", r.status_code)
print("Liczba rekordów Warszawa:",
      root.get("numberMatched"))
print("Liczba zwróconych:",
      root.get("numberReturned"))


Status: 200
Liczba rekordów Warszawa: 499944
Liczba zwróconych: 0


In [8]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET

url = "https://mapy.geoportal.gov.pl/wss/service/rcn"

filter_xml = """
<fes:Filter xmlns:fes="http://www.opengis.net/fes/2.0">
    <fes:PropertyIsEqualTo>
        <fes:ValueReference>TERYT</fes:ValueReference>
        <fes:Literal>1465</fes:Literal>
    </fes:PropertyIsEqualTo>
</fes:Filter>
"""

params = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "REQUEST": "GetFeature",
    "TYPENAMES": "ms:lokale",
    "COUNT": 300,
    "STARTINDEX": 0,
    "FILTER": filter_xml
}

r = requests.get(url, params=params, timeout=120)
r.raise_for_status()

root = ET.fromstring(r.content)

members = root.findall(
    ".//{http://www.opengis.net/wfs/2.0}member"
)

rows = []

for member in members:
    if len(member) == 0:
        continue

    feature = member[0]
    row = {}

    for element in feature:
        key = element.tag.split("}")[-1]

        if key == "GEOMETRIA":
            continue

        row[key] = element.text.strip() if element.text else None

    rows.append(row)

df = pd.DataFrame(rows)

print("Pobrano:", len(df), "rekordów")

if df.empty:
    print(r.text[:1500])
else:
    # Analiza rodzaju lokali
    print("\nRODZAJE LOKALI:")
    print(df["LOK_FUNKCJA"].value_counts(dropna=False))

    # Analiza dostępności danych
    print("\nDOSTĘPNOŚĆ DANYCH (%):")
    cols = [
        "LOK_CENA_BRUTTO",
        "LOK_POW_UZYT",
        "LOK_LICZBA_IZB",
        "LOK_NR_KOND",
        "DOK_DATA",
        "LOK_ADRES"
    ]

    print(
        (df[cols].notna().mean() * 100).round(1)
    )

    print("\nPRZYKŁADOWE TRANSAKCJE:")
    display(df[[
        "LOK_FUNKCJA",
        "LOK_CENA_BRUTTO",
        "LOK_POW_UZYT",
        "LOK_LICZBA_IZB",
        "LOK_NR_KOND",
        "DOK_DATA",
        "LOK_ADRES"
    ]].head(15))


Pobrano: 300 rekordów

RODZAJE LOKALI:
LOK_FUNKCJA
mieszkalna          242
garaz                50
handlowoUslugowa      4
inna                  3
biurowa               1
Name: count, dtype: int64

DOSTĘPNOŚĆ DANYCH (%):
LOK_CENA_BRUTTO    100.0
LOK_POW_UZYT       100.0
LOK_LICZBA_IZB     100.0
LOK_NR_KOND         99.0
DOK_DATA             0.0
LOK_ADRES          100.0
dtype: float64

PRZYKŁADOWE TRANSAKCJE:


,LOK_FUNKCJA,LOK_CENA_BRUTTO,LOK_POW_UZYT,LOK_LICZBA_IZB,LOK_NR_KOND,DOK_DATA,LOK_ADRES
0,mieszkalna,640000,34.2,2,1,None,MSC:Warszawa;UL:ulica Zawiszy;NR_PORZ:12
1,mieszkalna,995206.23,50.3,3,6,None,MSC:Warszawa;UL:ulica Namysłowska;NR_PORZ:6B
2,mieszkalna,600000,48.51,2,2,None,MSC:Warszawa;UL:ulica Leona Berensona;NR_PORZ:12E
3,mieszkalna,870000,37.24,2,6,None,MSC:Warszawa;UL:ulica Literacka;NR_PORZ:7A
4,mieszkalna,760000,44.58,3,4,None,MSC:Warszawa;UL:ulica Obrońców Tobruku;NR_PORZ:27
5,mieszkalna,495000,23.76,1,15,None,MSC:Warszawa;UL:ulica Puławska;NR_PORZ:7/9
6,mieszkalna,800000,56.8,2,4,None,MSC:Warszawa;UL:ulica Biruty;NR_PORZ:8
7,mieszkalna,1715645.08,96.71,4,3,None,MSC:Warszawa;UL:ulica Domaniewska;NR_PORZ:41B
8,mieszkalna,500000,24.27,1,2,None,MSC:Warszawa;UL:ulica Czerska;NR_PORZ:18
9,mieszkalna,825000,46.55,2,9,None,MSC:Warszawa;UL:ulica Puławska;NR_PORZ:233


In [9]:

import pandas as pd
import xml.etree.ElementTree as ET

rows = []

for member in members:
    if len(member) == 0:
        continue

    feature = member[0]
    row = {}

    for element in feature:
        key = element.tag.split("}")[-1]

        if key == "GEOMETRIA":
            continue

        # Odczyt również z zagnieżdżonych elementów XML
        value = "".join(element.itertext()).strip()
        row[key] = value if value else None

    rows.append(row)

df_check = pd.DataFrame(rows)

print("LICZBA REKORDÓW:", len(df_check))

print("\nRODZAJE LOKALI:")
print(df_check["LOK_FUNKCJA"].value_counts(dropna=False))

print("\nRODZAJE RYNKU:")
print(df_check["TRAN_RODZAJ_RYNKU"].value_counts(dropna=False))

print("\nKOMPLETNOŚĆ DANYCH:")
cols = [
    "LOK_CENA_BRUTTO",
    "LOK_POW_UZYT",
    "LOK_LICZBA_IZB",
    "LOK_NR_KOND",
    "DOK_DATA",
    "LOK_ADRES",
    "TRAN_RODZAJ_RYNKU"
]

display(
    (df_check[cols].notna().mean() * 100)
    .round(1)
    .to_frame("Dostępność (%)")
)

print("\nPRZYKŁADOWE NIEPUSTE DATY:")
print(df_check["DOK_DATA"].dropna().head(10).tolist())

# Sprawdzenie surowego pola daty w XML
for element in members[0][0]:
    if element.tag.split("}")[-1] == "DOK_DATA":
        print("\nSurowe pole DOK_DATA:")
        print(ET.tostring(element, encoding="unicode"))
        break


LICZBA REKORDÓW: 300

RODZAJE LOKALI:
LOK_FUNKCJA
mieszkalna          242
garaz                50
handlowoUslugowa      4
inna                  3
biurowa               1
Name: count, dtype: int64

RODZAJE RYNKU:
TRAN_RODZAJ_RYNKU
rynekWtorny       188
rynekPierwotny    112
Name: count, dtype: int64

KOMPLETNOŚĆ DANYCH:


,Dostępność (%)
LOK_CENA_BRUTTO,100.0
LOK_POW_UZYT,100.0
LOK_LICZBA_IZB,100.0
LOK_NR_KOND,99.0
DOK_DATA,100.0
LOK_ADRES,100.0
TRAN_RODZAJ_RYNKU,100.0



PRZYKŁADOWE NIEPUSTE DATY:
['2026-03-27', '2025-12-09', '2026-02-05', '2026-02-19', '2026-02-02', '2026-02-13', '2026-02-24', '2026-01-14', '2026-01-29', '2026-01-27']

Surowe pole DOK_DATA:
<ns0:DOK_DATA xmlns:ns0="http://mapserver.gis.umn.edu/mapserver" xmlns:ns1="http://www.opengis.net/gml/3.2" ns1:id="lokale.53969066.DOK_DATA"><ns1:timePosition>2026-03-27</ns1:timePosition></ns0:DOK_DATA>
        


In [10]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET
import time
from pathlib import Path

URL = "https://mapy.geoportal.gov.pl/wss/service/rcn"

LIMIT = 20000
PAGE_SIZE = 500

raw_dir = Path("data/raw/rcn_pages")
raw_dir.mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

filter_xml = """
<fes:Filter xmlns:fes="http://www.opengis.net/fes/2.0">
  <fes:PropertyIsEqualTo>
    <fes:ValueReference>TERYT</fes:ValueReference>
    <fes:Literal>1465</fes:Literal>
  </fes:PropertyIsEqualTo>
</fes:Filter>
"""

session = requests.Session()
all_rows = []
seen_ids = set()

for start in range(0, LIMIT, PAGE_SIZE):

    params = {
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "TYPENAMES": "ms:lokale",
        "COUNT": min(PAGE_SIZE, LIMIT - start),
        "STARTINDEX": start,
        "FILTER": filter_xml
    }

    r = session.get(
        URL, params=params, timeout=(15, 120)
    )
    r.raise_for_status()

    root = ET.fromstring(r.content)

    if not root.tag.endswith("FeatureCollection"):
        raise ValueError(r.text[:1000])

    members = root.findall(
        ".//{http://www.opengis.net/wfs/2.0}member"
    )

    if not members:
        print("Koniec dostępnych rekordów.")
        break

    # Zachowujemy oryginalną odpowiedź z geometrią
    (raw_dir / f"page_{start:06d}.gml").write_bytes(
        r.content
    )

    new_rows = []

    for member in members:
        if len(member) == 0:
            continue

        feature = member[0]
        row = {
            "feature_id": feature.get(
                "{http://www.opengis.net/gml/3.2}id"
            )
        }

        for element in feature:
            key = element.tag.split("}")[-1]

            if key == "GEOMETRIA":
                continue

            value = "".join(element.itertext()).strip()
            row[key] = value or None

        if row["feature_id"] in seen_ids:
            continue

        seen_ids.add(row["feature_id"])
        new_rows.append(row)

    if not new_rows:
        raise ValueError(
            "API zwróciło powtórzoną stronę. "
            "Przerwano pobieranie."
        )

    all_rows.extend(new_rows)

    print(
        f"StartIndex: {start} | "
        f"Nowe rekordy: {len(new_rows)} | "
        f"Łącznie: {len(all_rows)}"
    )

    time.sleep(1.5)

# Pełny zbiór
df_raw = pd.DataFrame(all_rows)

df_raw.to_csv(
    "data/raw/rcn_warszawa_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

# Tylko lokale mieszkalne
df_mieszkania = df_raw[
    df_raw["LOK_FUNKCJA"]
    .str.strip()
    .str.lower()
    .eq("mieszkalna")
].copy()

# Konwersja typów
numeric_cols = [
    "LOK_CENA_BRUTTO",
    "LOK_POW_UZYT",
    "LOK_LICZBA_IZB",
    "LOK_NR_KOND"
]

for col in numeric_cols:
    df_mieszkania[col] = pd.to_numeric(
        df_mieszkania[col].str.replace(
            ",", ".", regex=False
        ),
        errors="coerce"
    )

df_mieszkania["DOK_DATA"] = pd.to_datetime(
    df_mieszkania["DOK_DATA"],
    errors="coerce"
)

df_mieszkania.to_csv(
    "data/processed/rcn_warszawa_mieszkania.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nPODSUMOWANIE")
print("Wszystkie lokale:", len(df_raw))
print("Lokale mieszkalne:", len(df_mieszkania))
print("Najstarsza data:", df_mieszkania["DOK_DATA"].min())
print("Najnowsza data:", df_mieszkania["DOK_DATA"].max())

display(df_mieszkania.head(10))


StartIndex: 0 | Nowe rekordy: 500 | Łącznie: 500
StartIndex: 500 | Nowe rekordy: 312 | Łącznie: 812
StartIndex: 1000 | Nowe rekordy: 242 | Łącznie: 1054
StartIndex: 1500 | Nowe rekordy: 499 | Łącznie: 1553
StartIndex: 2000 | Nowe rekordy: 349 | Łącznie: 1902
StartIndex: 2500 | Nowe rekordy: 339 | Łącznie: 2241
StartIndex: 3000 | Nowe rekordy: 183 | Łącznie: 2424
StartIndex: 3500 | Nowe rekordy: 434 | Łącznie: 2858
StartIndex: 4000 | Nowe rekordy: 460 | Łącznie: 3318
StartIndex: 4500 | Nowe rekordy: 302 | Łącznie: 3620
StartIndex: 5000 | Nowe rekordy: 257 | Łącznie: 3877
StartIndex: 5500 | Nowe rekordy: 494 | Łącznie: 4371
StartIndex: 6000 | Nowe rekordy: 499 | Łącznie: 4870
StartIndex: 6500 | Nowe rekordy: 491 | Łącznie: 5361
StartIndex: 7000 | Nowe rekordy: 375 | Łącznie: 5736
StartIndex: 7500 | Nowe rekordy: 395 | Łącznie: 6131
StartIndex: 8000 | Nowe rekordy: 304 | Łącznie: 6435
StartIndex: 8500 | Nowe rekordy: 303 | Łącznie: 6738
StartIndex: 9000 | Nowe rekordy: 480 | Łącznie: 7218

,feature_id,boundedBy,gid,SERWIS_RCN,TERYT,TRAN_PRZESTRZEN_NAZW,TRAN_LOKALNY_ID_IIP,TRAN_WERSJA_ID,TRAN_OZNACZENIE_TRANS,TRAN_RODZAJ_TRANS,...,LOK_ID_LOKALU,LOK_FUNKCJA,LOK_LICZBA_IZB,LOK_NR_KOND,LOK_POW_UZYT,LOK_POW_PRZYN,LOK_CENA_BRUTTO,LOK_VAT,LOK_INFO,LOK_ADRES
0,lokale.53969066,488259.854285 634022.022566\n \t\t48825...,53969066,None,1465,PL.PZGiK.5346.RCN,76d63aec-605f-4a51-95da-57a4f4d50d75,2026-09-03T13:05:00Z,633535,wolnyRynek,...,146518_8.0307.46_BUD.26_LOK,mieszkalna,2,1.0,34.20,None,640000.00,None,None,MSC:Warszawa;UL:ulica Zawiszy;NR_PORZ:12
1,lokale.53961717,490625.688404 638892.328732\n \t\t49062...,53961717,None,1465,PL.PZGiK.5346.RCN,c1f9fa4f-65a4-4f00-8513-d485ab89da60,2026-07-07T12:36:05Z,616332,wolnyRynek,...,146508_8.1202.137_BUD.194_LOK,mieszkalna,3,6.0,50.30,None,995206.23,None,None,MSC:Warszawa;UL:ulica Namysłowska;NR_PORZ:6B
2,lokale.53966892,497243.848902 639894.967792\n \t\t49724...,53966892,None,1465,PL.PZGiK.5346.RCN,ffda4dfc-8ba2-430f-be64-5d09e0461b80,2026-06-23T15:27:58Z,626293,wolnyRynek,...,146503_8.1632.149_BUD.15_LOK,mieszkalna,2,2.0,48.51,None,600000.00,None,None,MSC:Warszawa;UL:ulica Leona Berensona;NR_PORZ:12E
3,lokale.53963485,490480.073396 633147.454823\n \t\t49048...,53963485,None,1465,PL.PZGiK.5346.RCN,25f5daa0-354d-4020-bf47-b729ac6aa879,2026-07-02T13:31:34Z,626414,wolnyRynek,...,146504_8.0604.83_BUD.33_LOK,mieszkalna,2,6.0,37.24,None,870000.00,None,None,MSC:Warszawa;UL:ulica Literacka;NR_PORZ:7A
4,lokale.53965811,489432.243419 631826.896336\n \t\t48943...,53965811,None,1465,PL.PZGiK.5346.RCN,e5cb2d16-e8aa-4586-b182-9988c66bb31c,2026-06-22T09:00:37Z,626647,wolnyRynek,...,146502_8.1501.385_BUD.91_LOK,mieszkalna,3,4.0,44.58,None,760000.00,None,None,MSC:Warszawa;UL:ulica Obrońców Tobruku;NR_PORZ:27
5,lokale.53968280,484800.392710 638043.455866\n \t\t48480...,53968280,None,1465,PL.PZGiK.5346.RCN,599ff654-b72c-41dc-8ff4-fcf67b3d4df6,2026-06-22T09:27:07Z,626658,wolnyRynek,...,146505_8.0113.33_BUD.78_LOK,mieszkalna,1,15.0,23.76,None,495000.00,None,None,MSC:Warszawa;UL:ulica Puławska;NR_PORZ:7/9
6,lokale.53969527,491165.625630 639191.035360\n \t\t49116...,53969527,None,1465,PL.PZGiK.5346.RCN,7c4d4bc2-57a3-4957-8eaa-9d0d78a5dbf6,2026-06-22T09:58:39Z,626659,wolnyRynek,...,146511_8.1012.1_BUD.116_LOK,mieszkalna,2,4.0,56.80,None,800000.00,None,None,MSC:Warszawa;UL:ulica Biruty;NR_PORZ:8
7,lokale.53963670,481510.311641 636629.752068\n \t\t48151...,53963670,None,1465,PL.PZGiK.5346.RCN,72b76384-166c-4833-ac03-32a313a4e993,2026-06-22T10:09:27Z,626675,wolnyRynek,...,146505_8.0809.1025_BUD.42_LOK,mieszkalna,4,3.0,96.71,2.42,1715645.08,127084.82,None,MSC:Warszawa;UL:ulica Domaniewska;NR_PORZ:41B
8,lokale.53965553,484130.403761 639806.265436\n \t\t48413...,53965553,None,1465,PL.PZGiK.5346.RCN,5c673ef6-f94d-40cd-8bf1-3988cb039479,2026-06-22T10:37:45Z,626687,wolnyRynek,...,146505_8.0306.29_BUD.453_LOK,mieszkalna,1,2.0,24.27,12.31,500000.00,None,None,MSC:Warszawa;UL:ulica Czerska;NR_PORZ:18
9,lokale.53969232,480685.772165 638166.130385\n \t\t48068...,53969232,None,1465,PL.PZGiK.5346.RCN,60ca0f71-5aad-4dbb-9900-122dec10bebe,2026-06-22T11:21:48Z,626697,wolnyRynek,...,146505_8.0407.60_BUD.123_LOK,mieszkalna,2,9.0,46.55,None,825000.00,None,None,MSC:Warszawa;UL:ulica Puławska;NR_PORZ:233


In [11]:

from pathlib import Path
from collections import Counter
import xml.etree.ElementTree as ET

files = sorted(Path("data/raw/rcn_pages").glob("*.gml"))

ids = []
gml_id = "{http://www.opengis.net/gml/3.2}id"
member_tag = "{http://www.opengis.net/wfs/2.0}member"

for file in files:
    root = ET.parse(file).getroot()
    members = root.findall(f".//{member_tag}")

    for member in members:
        if len(member):
            ids.append(member[0].get(gml_id))

counts = Counter(ids)

duplicates = sum(n - 1 for n in counts.values() if n > 1)

print("Liczba pobranych stron:", len(files))
print("Wszystkie zwrócone rekordy:", len(ids))
print("Unikalne identyfikatory:", len(counts))
print("Powtórzone rekordy:", duplicates)
print("ID występujące więcej niż raz:",
      sum(n > 1 for n in counts.values()))


Liczba pobranych stron: 40
Wszystkie zwrócone rekordy: 14049
Unikalne identyfikatory: 14025
Powtórzone rekordy: 24
ID występujące więcej niż raz: 24


In [12]:

import pandas as pd

df_eda = df_mieszkania.copy()

df_eda["DOK_DATA"] = pd.to_datetime(
    df_eda["DOK_DATA"],
    errors="coerce"
)

df_eda["year"] = df_eda["DOK_DATA"].dt.year

df_eda["price_per_m2"] = (
    df_eda["LOK_CENA_BRUTTO"]
    / df_eda["LOK_POW_UZYT"].replace(0, float("nan"))
)

print("MIESZKANIA WEDŁUG ROKU:")
display(
    df_eda["year"]
    .value_counts()
    .sort_index()
    .to_frame("Liczba mieszkań")
)

print("\nCENY ZA M2 – STATYSTYKI:")
display(
    df_eda["price_per_m2"]
    .describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])
)

print("\nRODZAJE RYNKU:")
display(
    df_eda["TRAN_RODZAJ_RYNKU"]
    .value_counts(dropna=False)
)

print("\nRODZAJE TRANSAKCJI:")
display(
    df_eda["TRAN_RODZAJ_TRANS"]
    .value_counts(dropna=False)
)


MIESZKANIA WEDŁUG ROKU:


,Liczba mieszkań
year,
2006,299
2007,296
2008,394
2009,392
2010,408
2011,446
2012,470
2013,530
2014,448



CENY ZA M2 – STATYSTYKI:


,price_per_m2
count,11339.000000
mean,10972.499914
std,5287.733108
min,37.878788
1%,2721.933968
25%,7193.511258
50%,9463.337393
75%,13898.702228
99%,27526.765739
max,67750.677507



RODZAJE RYNKU:


,count
TRAN_RODZAJ_RYNKU,
rynekWtorny,6918
rynekPierwotny,4470
None,4



RODZAJE TRANSAKCJI:


,count
TRAN_RODZAJ_TRANS,
wolnyRynek,5951
None,5428
sprzedazPrzetargowa,9
sprzedazZBonifikata,2
sprzedazWPostepowaniuEgzekucyjnym,2


In [13]:

import requests
import xml.etree.ElementTree as ET

URL = "https://mapy.geoportal.gov.pl/wss/service/rcn"

filter_xml = """
<fes:Filter xmlns:fes="http://www.opengis.net/fes/2.0">
  <fes:PropertyIsEqualTo>
    <fes:ValueReference>TERYT</fes:ValueReference>
    <fes:Literal>1465</fes:Literal>
  </fes:PropertyIsEqualTo>
</fes:Filter>
"""

start_index = 0
page_size = 500
all_ids = []

for page in range(5):

    params = {
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "TYPENAMES": "ms:lokale",
        "COUNT": page_size,
        "STARTINDEX": start_index,
        "SORTBY": "gid",
        "FILTER": filter_xml
    }

    r = requests.get(
        URL, params=params, timeout=120
    )
    r.raise_for_status()

    root = ET.fromstring(r.content)

    if not root.tag.endswith("FeatureCollection"):
        print("Błąd WFS:", r.text[:1000])
        break

    members = root.findall(
        ".//{http://www.opengis.net/wfs/2.0}member"
    )

    ids = []

    for member in members:
        if len(member) == 0:
            continue

        feature = member[0]
        gid = next(
            (
                e.text for e in feature
                if e.tag.split("}")[-1] == "gid"
            ),
            None
        )

        if gid is not None:
            ids.append(int(gid))

    print(
        f"Strona {page + 1}: "
        f"STARTINDEX={start_index}, "
        f"rekordów={len(members)}, "
        f"gid={min(ids) if ids else None}"
        f" ... {max(ids) if ids else None}"
    )

    all_ids.extend(ids)

    if not members:
        break

    # Przesuwamy się o faktyczną liczbę
    # zwróconych rekordów
    start_index += len(members)

print("\nPODSUMOWANIE")
print("Łącznie:", len(all_ids))
print("Unikalne ID:", len(set(all_ids)))
print("Duplikaty:", len(all_ids) - len(set(all_ids)))
print("Rosnąca kolejność ID:",
      all(a < b for a, b in zip(all_ids, all_ids[1:])))


Strona 1: STARTINDEX=0, rekordów=493, gid=23301067 ... 23301566
Strona 2: STARTINDEX=493, rekordów=487, gid=23301560 ... 23302059
Strona 3: STARTINDEX=980, rekordów=494, gid=23302047 ... 23302546
Strona 4: STARTINDEX=1474, rekordów=484, gid=23302541 ... 23303040
Strona 5: STARTINDEX=1958, rekordów=490, gid=23303025 ... 23303524

PODSUMOWANIE
Łącznie: 2448
Unikalne ID: 2406
Duplikaty: 42
Rosnąca kolejność ID: False


In [14]:

import requests
import xml.etree.ElementTree as ET

URL = "https://mapy.geoportal.gov.pl/wss/service/rcn"

all_ids = []

for page in range(5):

    start_index = page * 500

    params = {
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "TYPENAMES": "ms:lokale",
        "COUNT": 500,
        "STARTINDEX": start_index,
        "SORTBY": "gid",
        "FILTER": filter_xml
    }

    r = requests.get(
        URL, params=params, timeout=120
    )
    r.raise_for_status()

    root = ET.fromstring(r.content)

    members = root.findall(
        ".//{http://www.opengis.net/wfs/2.0}member"
    )

    ids = []

    for member in members:
        if len(member) == 0:
            continue

        feature = member[0]

        for element in feature:
            if element.tag.split("}")[-1] == "gid":
                ids.append(int(element.text))
                break

    all_ids.extend(ids)

    print(
        f"Strona {page+1}: "
        f"STARTINDEX={start_index}, "
        f"rekordów={len(ids)}, "
        f"gid={min(ids) if ids else None} "
        f"do {max(ids) if ids else None}"
    )

print("\nPODSUMOWANIE")
print("Pobrano:", len(all_ids))
print("Unikalne:", len(set(all_ids)))
print("Duplikaty:", len(all_ids)-len(set(all_ids)))

print(
    "Rosnące gid:",
    all(a < b for a, b in zip(all_ids, all_ids[1:]))
)


Strona 1: STARTINDEX=0, rekordów=493, gid=23301067 do 23301566
Strona 2: STARTINDEX=500, rekordów=487, gid=23301567 do 23302066
Strona 3: STARTINDEX=1000, rekordów=494, gid=23302067 do 23302566
Strona 4: STARTINDEX=1500, rekordów=484, gid=23302567 do 23303066
Strona 5: STARTINDEX=2000, rekordów=489, gid=23303067 do 23303566

PODSUMOWANIE
Pobrano: 2447
Unikalne: 2447
Duplikaty: 0
Rosnące gid: True


In [17]:

import requests
import xml.etree.ElementTree as ET
import pandas as pd

URL = "https://mapy.geoportal.gov.pl/wss/service/rcn"

filter_xml = """
<fes:Filter xmlns:fes="http://www.opengis.net/fes/2.0">
  <fes:And>
    <fes:PropertyIsEqualTo>
      <fes:ValueReference>TERYT</fes:ValueReference>
      <fes:Literal>1465</fes:Literal>
    </fes:PropertyIsEqualTo>

    <fes:PropertyIsEqualTo>
      <fes:ValueReference>LOK_FUNKCJA</fes:ValueReference>
      <fes:Literal>mieszkalna</fes:Literal>
    </fes:PropertyIsEqualTo>

    <fes:PropertyIsGreaterThanOrEqualTo>
      <fes:ValueReference>DOK_DATA</fes:ValueReference>
      <fes:Literal>2023-01-01</fes:Literal>
    </fes:PropertyIsGreaterThanOrEqualTo>

    <fes:PropertyIsLessThan>
      <fes:ValueReference>DOK_DATA</fes:ValueReference>
      <fes:Literal>2027-01-01</fes:Literal>
    </fes:PropertyIsLessThan>
  </fes:And>
</fes:Filter>
"""

base_params = {
    "SERVICE": "WFS",
    "VERSION": "2.0.0",
    "REQUEST": "GetFeature",
    "TYPENAMES": "ms:lokale",
    "FILTER": filter_xml
}

# Najpierw liczba pasujących rekordów
r = requests.get(
    URL,
    params={**base_params, "RESULTTYPE": "hits"},
    timeout=90
)
r.raise_for_status()

root = ET.fromstring(r.content)

print("Status:", r.status_code)
print("Liczba pasujących mieszkań:", root.get("numberMatched"))

# Pobieramy 10 rekordów kontrolnych
r = requests.get(
    URL,
    params={
        **base_params,
        "COUNT": 10,
        "STARTINDEX": 0,
        "SORTBY": "gid"
    },
    timeout=90
)
r.raise_for_status()

root = ET.fromstring(r.content)
members = root.findall(
    ".//{http://www.opengis.net/wfs/2.0}member"
)

rows = []

for member in members:
    if len(member) == 0:
        continue

    feature = member[0]
    row = {}

    for element in feature:
        key = element.tag.split("}")[-1]

        if key in ("GEOMETRIA", "boundedBy"):
            continue

        value = "".join(element.itertext()).strip()
        row[key] = value or None

    rows.append(row)

df_test = pd.DataFrame(rows)

print("Pobrano próbkę:", len(df_test))

cols = [
    "TERYT",
    "LOK_FUNKCJA",
    "DOK_DATA",
    "LOK_CENA_BRUTTO",
    "LOK_POW_UZYT"
]

display(df_test.reindex(columns=cols))

# Kontrola działania filtra
if not df_test.empty:
    dates = pd.to_datetime(df_test["DOK_DATA"], errors="coerce")

    poprawne = (
        df_test["TERYT"].eq("1465")
        & df_test["LOK_FUNKCJA"].eq("mieszkalna")
        & dates.ge("2023-01-01")
        & dates.lt("2027-01-01")
    )

    print("Wszystkie rekordy spełniają filtr:", poprawne.all())


Status: 200
Liczba pasujących mieszkań: 89738
Pobrano próbkę: 10


,TERYT,LOK_FUNKCJA,DOK_DATA,LOK_CENA_BRUTTO,LOK_POW_UZYT
0,1465,mieszkalna,2023-01-10,558089,39.33
1,1465,mieszkalna,2023-01-12,1351000,122.48
2,1465,mieszkalna,2023-01-13,225310,23.82
3,1465,mieszkalna,2023-01-10,487600,57.2
4,1465,mieszkalna,2023-01-04,570000,43.87
5,1465,mieszkalna,2023-01-13,575781.25,65.87
6,1465,mieszkalna,2023-01-09,335000,19.95
7,1465,mieszkalna,2023-01-11,422829,25.12
8,1465,mieszkalna,2023-01-13,899986,34.22
9,1465,mieszkalna,2023-01-12,1054000,66


Wszystkie rekordy spełniają filtr: True


In [19]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [20]:

import requests
import pandas as pd
import xml.etree.ElementTree as ET
import gzip
import time
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

URL = "https://mapy.geoportal.gov.pl/wss/service/rcn"

EXPECTED = 89738
PAGE_SIZE = 500
MAX_PAGES_PER_RUN = 1200

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
RAW = BASE / "raw"
RAW.mkdir(parents=True, exist_ok=True)

filter_xml = """
<fes:Filter xmlns:fes="http://www.opengis.net/fes/2.0">
 <fes:And>
  <fes:PropertyIsEqualTo>
   <fes:ValueReference>TERYT</fes:ValueReference>
   <fes:Literal>1465</fes:Literal>
  </fes:PropertyIsEqualTo>
  <fes:PropertyIsEqualTo>
   <fes:ValueReference>LOK_FUNKCJA</fes:ValueReference>
   <fes:Literal>mieszkalna</fes:Literal>
  </fes:PropertyIsEqualTo>
  <fes:PropertyIsGreaterThanOrEqualTo>
   <fes:ValueReference>DOK_DATA</fes:ValueReference>
   <fes:Literal>2023-01-01</fes:Literal>
  </fes:PropertyIsGreaterThanOrEqualTo>
  <fes:PropertyIsLessThan>
   <fes:ValueReference>DOK_DATA</fes:ValueReference>
   <fes:Literal>2027-01-01</fes:Literal>
  </fes:PropertyIsLessThan>
 </fes:And>
</fes:Filter>
"""

def parse_records(content):
    root = ET.fromstring(content)

    if root.tag.split("}")[-1] != "FeatureCollection":
        raise ValueError(
            "Nieprawidlowa odpowiedz WFS: "
            + content[:300].decode("utf-8", "replace")
        )

    rows = []

    for member in root.findall(
        ".//{http://www.opengis.net/wfs/2.0}member"
    ):
        if len(member) == 0:
            continue

        feature = member[0]
        row = {}

        for element in feature:
            name = element.tag.split("}")[-1]

            if name in ("GEOMETRIA", "boundedBy"):
                continue

            value = "".join(element.itertext()).strip()
            row[name] = value or None

        if row.get("gid"):
            rows.append(row)

    return rows


# Odnajdujemy juz zapisane strony
saved = sorted(RAW.glob("page_*.gml.gz"))
seen = set()

for file in saved:
    with gzip.open(file, "rb") as f:
        rows = parse_records(f.read())
    seen.update(x["gid"] for x in rows)

if saved:
    last_start = int(
        saved[-1].name.split("_")[1].split(".")[0]
    )
    start_index = last_start + PAGE_SIZE
else:
    start_index = 0

print("Wczesniej zapisanych stron:", len(saved))
print("Unikalnych rekordow:", len(seen))
print("Wznawiam od STARTINDEX:", start_index)

session = requests.Session()

retry = Retry(
    total=4,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504]
)

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

for page in range(MAX_PAGES_PER_RUN):

    if len(seen) >= EXPECTED:
        print("Osiagnieto deklarowana liczbe rekordow.")
        break

    params = {
        "SERVICE": "WFS",
        "VERSION": "2.0.0",
        "REQUEST": "GetFeature",
        "TYPENAMES": "ms:lokale",
        "COUNT": PAGE_SIZE,
        "STARTINDEX": start_index,
        "SORTBY": "gid",
        "FILTER": filter_xml
    }

    try:
        response = session.get(
            URL,
            params=params,
            timeout=(20, 120)
        )
        response.raise_for_status()

        rows = parse_records(response.content)

    except (requests.RequestException, ValueError) as e:
        print("Blad pobierania:", e)
        print("Postep zapisany. Uruchom ponownie.")
        break

    ids = [row["gid"] for row in rows]

    # Zapisujemy pelny GML, lacznie z geometria
    output = RAW / f"page_{start_index:08d}.gml.gz"

    with gzip.open(output, "wb") as f:
        f.write(response.content)

    new_ids = set(ids) - seen
    seen.update(ids)

    if page % 10 == 0 or len(seen) >= EXPECTED:
        print(
            f"STARTINDEX={start_index} | "
            f"zwrócono={len(rows)} | "
            f"nowe={len(new_ids)} | "
            f"unikalne={len(seen)}/{EXPECTED}"
        )

    # Stale przesuniecie, NIE liczba zwroconych
    start_index += PAGE_SIZE

    time.sleep(0.5)

print("\nPODSUMOWANIE")
print("Unikalne rekordy:", len(seen))
print("Deklarowane przez API:", EXPECTED)
print("Nastepny STARTINDEX:", start_index)

if len(seen) == EXPECTED:
    print("Osiagnieto deklarowana liczbe.")
else:
    print(
        "Zbior nie jest jeszcze potwierdzony "
        "jako kompletny."
    )


Wczesniej zapisanych stron: 0
Unikalnych rekordow: 0
Wznawiam od STARTINDEX: 0
STARTINDEX=0 | zwrócono=500 | nowe=500 | unikalne=500/89738
STARTINDEX=5000 | zwrócono=500 | nowe=500 | unikalne=5500/89738
STARTINDEX=10000 | zwrócono=500 | nowe=500 | unikalne=10500/89738
STARTINDEX=15000 | zwrócono=500 | nowe=500 | unikalne=15500/89738
STARTINDEX=20000 | zwrócono=500 | nowe=500 | unikalne=20500/89738
STARTINDEX=25000 | zwrócono=500 | nowe=500 | unikalne=25500/89738
STARTINDEX=30000 | zwrócono=500 | nowe=500 | unikalne=30500/89738
STARTINDEX=35000 | zwrócono=500 | nowe=500 | unikalne=35500/89738
STARTINDEX=40000 | zwrócono=500 | nowe=500 | unikalne=40500/89738
STARTINDEX=45000 | zwrócono=500 | nowe=500 | unikalne=45500/89738
STARTINDEX=50000 | zwrócono=500 | nowe=500 | unikalne=50500/89738
STARTINDEX=55000 | zwrócono=500 | nowe=500 | unikalne=55500/89738
STARTINDEX=60000 | zwrócono=500 | nowe=500 | unikalne=60500/89738
STARTINDEX=65000 | zwrócono=500 | nowe=500 | unikalne=65500/89738
START

In [22]:

import pandas as pd
import gzip
from pathlib import Path

# Folder z danymi zapisanymi na Google Drive
BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
RAW = BASE / "raw"

rows = []

# Odczyt wszystkich zapisanych stron GML
for file in sorted(RAW.glob("page_*.gml.gz")):
    with gzip.open(file, "rb") as f:
        rows.extend(parse_records(f.read()))

# Tworzenie DataFrame
df = pd.DataFrame(rows)

print("Wszystkie odczytane rekordy:", len(df))

# Usuwanie duplikatów
df = df.drop_duplicates(subset=["gid"])

# Konwersja dat
df["DOK_DATA"] = pd.to_datetime(
    df["DOK_DATA"],
    errors="coerce"
)

# Filtrowanie mieszkań z Warszawy z lat 2023-2026
mask = (
    df["TERYT"].eq("1465")
    & df["LOK_FUNKCJA"].eq("mieszkalna")
    & df["DOK_DATA"].between(
        "2023-01-01",
        "2026-12-31"
    )
)

df = df.loc[mask].copy()

# Zapis CSV
output_path = BASE / "warszawa_mieszkania_2023_2026.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nPODSUMOWANIE")
print("Unikalne mieszkania:", len(df))
print("Plik zapisany:", output_path)

print("\nLICZBA MIESZKAŃ WEDŁUG ROKU:")
display(
    df["DOK_DATA"]
    .dt.year
    .value_counts()
    .sort_index()
    .to_frame("Liczba mieszkań")
)

print("\nPRZYKŁADOWE REKORDY:")
display(df.head(10))


Wszystkie odczytane rekordy: 89738

PODSUMOWANIE
Unikalne mieszkania: 89738
Plik zapisany: /content/drive/MyDrive/ZPA_RCN_Warszawa/warszawa_mieszkania_2023_2026.csv

LICZBA MIESZKAŃ WEDŁUG ROKU:


,Liczba mieszkań
DOK_DATA,
2023,23612
2024,24540
2025,31923
2026,9663



PRZYKŁADOWE REKORDY:


,gid,SERWIS_RCN,TERYT,TRAN_PRZESTRZEN_NAZW,TRAN_LOKALNY_ID_IIP,TRAN_WERSJA_ID,TRAN_OZNACZENIE_TRANS,TRAN_RODZAJ_TRANS,TRAN_RODZAJ_RYNKU,TRAN_SPRZEDAJACY,...,LOK_ID_LOKALU,LOK_FUNKCJA,LOK_LICZBA_IZB,LOK_NR_KOND,LOK_POW_UZYT,LOK_POW_PRZYN,LOK_CENA_BRUTTO,LOK_VAT,LOK_INFO,LOK_ADRES
0,23325006,None,1465,PL.PZGiK.5346.RCN,17d4f86c-da86-406c-a0fe-b8d497f2bdf7,2023-07-28T14:18:02Z,492063,wolnyRynek,rynekPierwotny,osobaPrawna,...,146513_8.0814.1/6.4_BUD.99_LOK,mieszkalna,2,7,39.33,None,558089,None,None,MSC:Warszawa;UL:ulica Bokserska;NR_PORZ:73
1,23325070,None,1465,PL.PZGiK.5346.RCN,e1f9e3c6-d7c0-481f-bb74-29d631920d37,2023-08-17T14:07:19Z,493422,wolnyRynek,rynekWtorny,osobaFizyczna,...,146511_8.0821.95/10.1_BUD.4_LOK,mieszkalna,4,1,122.48,None,1351000,100074.07,None,MSC:Warszawa;UL:ulica Budowlana;NR_PORZ:5
2,23325073,None,1465,PL.PZGiK.5346.RCN,a5d3008e-9f85-49f1-ad4d-462240957eeb,2026-02-06T15:57:35Z,493523,wolnyRynek,rynekWtorny,osobaFizyczna,...,146512_8.1107.28.1_BUD.1_LOK,mieszkalna,2,1,23.82,None,225310,None,None,MSC:Warszawa;UL:ulica Franciszka Adolfa Achera...
3,23325075,None,1465,PL.PZGiK.5346.RCN,1d084269-785b-4aa3-a1d1-e1558aac327a,2023-08-18T14:53:52Z,493640,wolnyRynek,rynekPierwotny,osobaPrawna,...,146512_8.0909.92/2.3_BUD.84_LOK,mieszkalna,3,4,57.2,None,487600,None,None,MSC:Warszawa;UL:ulica Herbu Oksza;NR_PORZ:3
4,23325082,None,1465,PL.PZGiK.5346.RCN,1d02400d-030a-4271-8857-28136fe837fb,2023-08-21T14:19:33Z,493814,wolnyRynek,rynekWtorny,osobaFizyczna,...,146512_8.1116.31/5.1_BUD.252_LOK,mieszkalna,2,3,43.87,None,570000,None,None,MSC:Warszawa;UL:ulica Dzieci Warszawy;NR_PORZ:17D
5,23325084,None,1465,PL.PZGiK.5346.RCN,a98c80e9-3f6b-40ad-8cc8-e8afbacc4656,2023-08-21T14:50:14Z,493837,wolnyRynek,rynekPierwotny,osobaPrawna,...,146512_8.0909.92/2.3_BUD.30_LOK,mieszkalna,4,4,65.87,None,575781.25,None,None,MSC:Warszawa;UL:ulica Herbu Oksza;NR_PORZ:3
6,23325088,None,1465,PL.PZGiK.5346.RCN,02c1e586-a311-42ec-8668-d747da31bfaf,2023-08-22T10:01:32Z,493913,wolnyRynek,rynekWtorny,osobaFizyczna,...,146518_8.0102.80.1_BUD.84_LOK,mieszkalna,1,6,19.95,None,335000,None,None,MSC:Warszawa;UL:ulica Ogrodowa;NR_PORZ:5
7,23325090,None,1465,PL.PZGiK.5346.RCN,01d8989f-fca8-484e-a7d3-10159b000bde,2023-08-22T10:14:57Z,493920,wolnyRynek,rynekWtorny,osobaFizyczna,...,146518_8.0302.39/12.1_BUD.182_LOK,mieszkalna,1,2,25.12,None,422829,None,None,MSC:Warszawa;UL:ulica Burakowska;NR_PORZ:16B
8,23325091,None,1465,PL.PZGiK.5346.RCN,ccdc83f1-c7ab-4cf0-ab69-da5615c11b68,2023-08-22T13:00:15Z,493993,wolnyRynek,rynekWtorny,osobaFizyczna,...,146518_8.0106.110/3.6_BUD.27_LOK,mieszkalna,2,5,34.22,None,899986,None,None,MSC:Warszawa;UL:ulica Waliców;NR_PORZ:11
9,23325092,None,1465,PL.PZGiK.5346.RCN,d791c0d1-7fbd-4110-9a4d-baf26aa3c9d8,2023-08-22T14:00:50Z,494010,wolnyRynek,rynekWtorny,osobaFizyczna,...,146518_8.0308.4/9.10_BUD.177_LOK,mieszkalna,4,1,66,None,1054000,None,None,MSC:Warszawa;UL:ulica św. Stanisława;NR_PORZ:1A


In [23]:
import gzip
import xml.etree.ElementTree as ET
from pathlib import Path

RAW = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa/raw")

files = sorted(RAW.glob("page_*.gml.gz"))

with gzip.open(files[0], "rb") as f:
    root = ET.fromstring(f.read())

members = root.findall(
    ".//{http://www.opengis.net/wfs/2.0}member"
)

feature = members[0][0]

geometry = next(
    (
        el for el in feature
        if el.tag.split("}")[-1] == "GEOMETRIA"
    ),
    None
)

print("GEOMETRIA PIERWSZEGO MIESZKANIA:")

if geometry is not None:
    print(
        ET.tostring(
            geometry,
            encoding="unicode"
        )[:3000]
    )

    print("\nUKŁAD WSPÓŁRZĘDNYCH:")
    for el in geometry.iter():
        if "srsName" in el.attrib:
            print(el.attrib["srsName"])

        if el.tag.split("}")[-1] in (
            "pos", "posList", "coordinates"
        ):
            print("Współrzędne:", el.text[:300])
else:
    print("Nie znaleziono geometrii.")

GEOMETRIA PIERWSZEGO MIESZKANIA:
<ns0:GEOMETRIA xmlns:ns0="http://mapserver.gis.umn.edu/mapserver" xmlns:ns1="http://www.opengis.net/gml/3.2">
          <ns1:Point ns1:id="lokale.23325006.1" srsName="urn:ogc:def:crs:EPSG::2180">
            <ns1:pos>480194.064028 636007.779307</ns1:pos>
          </ns1:Point>
        </ns0:GEOMETRIA>
        

UKŁAD WSPÓŁRZĘDNYCH:
urn:ogc:def:crs:EPSG::2180
Współrzędne: 480194.064028 636007.779307


In [24]:
%pip install -q pyproj

In [25]:

import gzip
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from pyproj import Transformer

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
RAW = BASE / "raw"

files = sorted(RAW.glob("page_*.gml.gz"))

if not files:
    raise FileNotFoundError("Nie znaleziono plikow GML")

print("Liczba plikow GML:", len(files))

# 1. Odczyt geometrii i identyfikatorow
records = []

NS_WFS = "{http://www.opengis.net/wfs/2.0}"
NS_GML = "{http://www.opengis.net/gml/3.2}"

for i, file in enumerate(files, 1):

    with gzip.open(file, "rb") as f:
        root = ET.fromstring(f.read())

    for member in root.findall(f".//{NS_WFS}member"):
        if len(member) == 0:
            continue

        feature = member[0]

        gid = next(
            (
                el.text for el in feature
                if el.tag.split("}")[-1] == "gid"
            ),
            None
        )

        geometry = next(
            (
                el for el in feature
                if el.tag.split("}")[-1] == "GEOMETRIA"
            ),
            None
        )

        northing = np.nan
        easting = np.nan

        if geometry is not None:
            point = geometry.find(f".//{NS_GML}Point")

            if point is not None:
                srs = point.get("srsName", "")
                pos = point.find(f"{NS_GML}pos")

                if (
                    "2180" in srs
                    and pos is not None
                    and pos.text
                ):
                    values = pos.text.split()

                    if len(values) >= 2:
                        # EPSG:2180: northing, easting
                        northing = float(values[0])
                        easting = float(values[1])

        if gid is not None:
            records.append({
                "gid": str(gid),
                "northing_2180": northing,
                "easting_2180": easting
            })

    if i % 20 == 0:
        print(f"Przetworzono plikow: {i}/{len(files)}")

geo = pd.DataFrame(records)

print("\nOdczytane rekordy:", len(geo))
print("Duplikaty gid:", geo["gid"].duplicated().sum())

geo = geo.drop_duplicates(subset=["gid"]).copy()

# 2. Konwersja EPSG:2180 -> GPS EPSG:4326
transformer = Transformer.from_crs(
    "EPSG:2180",
    "EPSG:4326",
    always_xy=True
)

valid = geo[
    ["northing_2180", "easting_2180"]
].notna().all(axis=1)

geo["latitude"] = np.nan
geo["longitude"] = np.nan

# always_xy=True wymaga easting przed northing
lon, lat = transformer.transform(
    geo.loc[valid, "easting_2180"].to_numpy(),
    geo.loc[valid, "northing_2180"].to_numpy()
)

geo.loc[valid, "longitude"] = lon
geo.loc[valid, "latitude"] = lat

# 3. Kontrola lokalizacji
geo["warsaw_location_ok"] = (
    geo["latitude"].between(51.9, 52.5)
    & geo["longitude"].between(20.7, 21.4)
)

print("\nPoprawne wspolrzedne:", valid.sum())
print("W obszarze kontrolnym Warszawy:",
      geo["warsaw_location_ok"].sum())
print("Poza obszarem kontrolnym:",
      (valid & ~geo["warsaw_location_ok"]).sum())

# 4. Polaczenie z glownym CSV
source_path = BASE / "warszawa_mieszkania_2023_2026.csv"

apartments = pd.read_csv(
    source_path,
    dtype={"gid": "string"}
)

geo["gid"] = geo["gid"].astype("string")

df_geo = apartments.merge(
    geo,
    on="gid",
    how="left",
    validate="one_to_one"
)

# 5. Odleglosc od umownego centrum Warszawy
# Palac Kultury i Nauki
CENTER_LAT = 52.23184
CENTER_LON = 21.00673

R = 6371.0088

lat1 = np.radians(df_geo["latitude"].astype(float))
lon1 = np.radians(df_geo["longitude"].astype(float))
lat2 = np.radians(CENTER_LAT)
lon2 = np.radians(CENTER_LON)

a = (
    np.sin((lat2 - lat1) / 2) ** 2
    + np.cos(lat1) * np.cos(lat2)
    * np.sin((lon2 - lon1) / 2) ** 2
)

df_geo["distance_center_km"] = (
    2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
)

# 6. Zapis polaczonego datasetu
output = BASE / "warszawa_mieszkania_geo.csv"

df_geo.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\nPODSUMOWANIE")
print("Mieszkania:", len(df_geo))
print("Ze wspolrzednymi:",
      df_geo["latitude"].notna().sum())
print("Bez wspolrzednych:",
      df_geo["latitude"].isna().sum())
print("Zapisano:", output)

display(df_geo[[
    "gid",
    "LOK_ADRES",
    "latitude",
    "longitude",
    "distance_center_km"
]].head(10))


Liczba plikow GML: 180
Przetworzono plikow: 20/180
Przetworzono plikow: 40/180
Przetworzono plikow: 60/180
Przetworzono plikow: 80/180
Przetworzono plikow: 100/180
Przetworzono plikow: 120/180
Przetworzono plikow: 140/180
Przetworzono plikow: 160/180
Przetworzono plikow: 180/180

Odczytane rekordy: 89738
Duplikaty gid: 0

Poprawne wspolrzedne: 89738
W obszarze kontrolnym Warszawy: 89738
Poza obszarem kontrolnym: 0

PODSUMOWANIE
Mieszkania: 89738
Ze wspolrzednymi: 89738
Bez wspolrzednych: 0
Zapisano: /content/drive/MyDrive/ZPA_RCN_Warszawa/warszawa_mieszkania_geo.csv


,gid,LOK_ADRES,latitude,longitude,distance_center_km
0,23325006,MSC:Warszawa;UL:ulica Bokserska;NR_PORZ:73,52.171052,20.989446,6.861157
1,23325070,MSC:Warszawa;UL:ulica Budowlana;NR_PORZ:5,52.280831,21.024295,5.577214
2,23325073,MSC:Warszawa;UL:ulica Franciszka Adolfa Achera...,52.197188,20.890053,8.833854
3,23325075,MSC:Warszawa;UL:ulica Herbu Oksza;NR_PORZ:3,52.199516,20.887291,8.895669
4,23325082,MSC:Warszawa;UL:ulica Dzieci Warszawy;NR_PORZ:17D,52.189767,20.908775,8.150563
5,23325084,MSC:Warszawa;UL:ulica Herbu Oksza;NR_PORZ:3,52.199516,20.887291,8.895669
6,23325088,MSC:Warszawa;UL:ulica Ogrodowa;NR_PORZ:5,52.239551,20.993422,1.247603
7,23325090,MSC:Warszawa;UL:ulica Burakowska;NR_PORZ:16B,52.256323,20.980202,3.267078
8,23325091,MSC:Warszawa;UL:ulica Waliców;NR_PORZ:11,52.234040,20.992059,1.028600
9,23325092,MSC:Warszawa;UL:ulica św. Stanisława;NR_PORZ:1A,52.246901,20.963312,3.397789


In [26]:
%pip install -q geopandas

In [27]:

import requests
import geopandas as gpd
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")

# 1. Wczytanie mieszkań ze współrzędnymi
df = pd.read_csv(
    BASE / "warszawa_mieszkania_geo.csv",
    dtype={"gid": "string"}
)

print("Mieszkania:", len(df))

# 2. Pobranie granic dzielnic przez ArcGIS API
url = (
    "https://services.arcgis.com/"
    "VU1dOESuQS3Z8oTW/ArcGIS/rest/services/"
    "baza__WTP_widok/FeatureServer/4/query"
)

params = {
    "where": "czyWarszawa='TAK'",
    "outFields": "Nazwa,czyWarszawa",
    "returnGeometry": "true",
    "outSR": 4326,
    "f": "geojson"
}

r = requests.get(url, params=params, timeout=90)
r.raise_for_status()

data = r.json()

if "error" in data:
    raise ValueError(data["error"])

if not data.get("features"):
    raise ValueError("API nie zwrocilo granic dzielnic")

districts = gpd.GeoDataFrame.from_features(
    data["features"],
    crs="EPSG:4326"
)

districts = districts[
    districts["czyWarszawa"] == "TAK"
].copy()

print("Liczba dzielnic:", districts["Nazwa"].nunique())
print(sorted(districts["Nazwa"].dropna().unique()))

if districts["Nazwa"].nunique() != 18:
    raise ValueError(
        "Nie otrzymano 18 dzielnic. "
        "Sprawdz dane z API przed laczeniem."
    )

# 3. Zamiana wspolrzednych mieszkan na punkty
points = gpd.GeoDataFrame(
    df[["gid"]].copy(),
    geometry=gpd.points_from_xy(
        df["longitude"],
        df["latitude"]
    ),
    crs="EPSG:4326"
)

# 4. Przypisanie mieszkan do dzielnic
joined = gpd.sjoin(
    points,
    districts[["Nazwa", "geometry"]],
    how="left",
    predicate="within"
)

if joined["gid"].duplicated().any():
    raise ValueError(
        "Niektore mieszkania pasuja do wielu poligonow."
    )

# 5. Dolaczenie nazwy dzielnicy do datasetu
district_map = joined.set_index("gid")["Nazwa"]

df["district"] = df["gid"].map(district_map)

# 6. Zapis
output = BASE / "warszawa_mieszkania_dzielnice.csv"

df.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\nPODSUMOWANIE")
print("Wszystkie mieszkania:", len(df))
print("Przypisana dzielnica:", df["district"].notna().sum())
print("Brak dzielnicy:", df["district"].isna().sum())

print("\nLICZBA MIESZKAN WEDLUG DZIELNIC:")
display(df["district"].value_counts(dropna=False))

print("\nPRZYKLADOWE REKORDY:")
display(df[[
    "LOK_ADRES",
    "latitude",
    "longitude",
    "district",
    "distance_center_km"
]].head(10))


Mieszkania: 89738
Liczba dzielnic: 18
['Bemowo', 'Białołęka', 'Bielany', 'Mokotów', 'Ochota', 'Praga Południe', 'Praga Północ', 'Rembertów', 'Targówek', 'Ursus', 'Ursynów', 'Wawer', 'Wesoła', 'Wilanów', 'Wola', 'Włochy', 'Śródmieście', 'Żoliborz']

PODSUMOWANIE
Wszystkie mieszkania: 89738
Przypisana dzielnica: 89738
Brak dzielnicy: 0

LICZBA MIESZKAN WEDLUG DZIELNIC:


,count
district,
Mokotów,11632
Białołęka,10863
Wola,9192
Praga Południe,8359
Ursus,6544
Bemowo,5954
Włochy,5349
Śródmieście,4047
Bielany,3931



PRZYKLADOWE REKORDY:


,LOK_ADRES,latitude,longitude,district,distance_center_km
0,MSC:Warszawa;UL:ulica Bokserska;NR_PORZ:73,52.171052,20.989446,Ursynów,6.861157
1,MSC:Warszawa;UL:ulica Budowlana;NR_PORZ:5,52.280831,21.024295,Targówek,5.577214
2,MSC:Warszawa;UL:ulica Franciszka Adolfa Achera...,52.197188,20.890053,Ursus,8.833854
3,MSC:Warszawa;UL:ulica Herbu Oksza;NR_PORZ:3,52.199516,20.887291,Ursus,8.895669
4,MSC:Warszawa;UL:ulica Dzieci Warszawy;NR_PORZ:17D,52.189767,20.908775,Ursus,8.150563
5,MSC:Warszawa;UL:ulica Herbu Oksza;NR_PORZ:3,52.199516,20.887291,Ursus,8.895669
6,MSC:Warszawa;UL:ulica Ogrodowa;NR_PORZ:5,52.239551,20.993422,Wola,1.247603
7,MSC:Warszawa;UL:ulica Burakowska;NR_PORZ:16B,52.256323,20.980202,Wola,3.267078
8,MSC:Warszawa;UL:ulica Waliców;NR_PORZ:11,52.234040,20.992059,Wola,1.028600
9,MSC:Warszawa;UL:ulica św. Stanisława;NR_PORZ:1A,52.246901,20.963312,Wola,3.397789


In [28]:
%pip install -q osmnx geopandas shapely

In [29]:

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
OSM_DIR = BASE / "osm"
OSM_DIR.mkdir(parents=True, exist_ok=True)

# Ustawienia API i cache
ox.settings.use_cache = True
ox.settings.cache_folder = str(BASE / "osm_cache")
ox.settings.requests_timeout = 180
ox.settings.overpass_rate_limit = True

# Nasz dataset z dzielnicami
df = pd.read_csv(
    BASE / "warszawa_mieszkania_dzielnice.csv",
    dtype={"gid": "string"}
)

# Obszar z marginesem ok. 1.5 km
south = df["latitude"].min() - 0.014
north = df["latitude"].max() + 0.014
west = df["longitude"].min() - 0.023
east = df["longitude"].max() + 0.023

# 3 x 3 mniejsze obszary
lat_edges = np.linspace(south, north, 4)
lon_edges = np.linspace(west, east, 4)

tags = {
    "shop": True,
    "leisure": "park"
}

for i in range(3):
    for j in range(3):

        path = OSM_DIR / f"osm_tile_{i}_{j}.gpkg"

        if path.exists():
            print(f"Fragment {i},{j}: już zapisany")
            continue

        bbox = (
            float(lon_edges[j]),
            float(lat_edges[i]),
            float(lon_edges[j + 1]),
            float(lat_edges[i + 1])
        )

        print(f"Pobieram fragment {i},{j}...")

        poi = ox.features_from_bbox(
            bbox,
            tags=tags
        )

        # Zachowujemy identyfikator OSM, typ i geometrię
        part = gpd.GeoDataFrame(
            {
                "osm_key": [
                    f"{kind}/{osmid}"
                    for kind, osmid in poi.index
                ],
                "shop": (
                    poi["shop"].astype("string").fillna("")
                    if "shop" in poi.columns
                    else [""] * len(poi)
                ),
                "leisure": (
                    poi["leisure"].astype("string").fillna("")
                    if "leisure" in poi.columns
                    else [""] * len(poi)
                )
            },
            geometry=list(poi.geometry),
            crs=poi.crs
        )

        part.to_file(
            path,
            layer="pois",
            driver="GPKG"
        )

        print("Zapisano obiektów:", len(part))

print("\nPobieranie fragmentów zakończone.")


Pobieram fragment 0,0...
Zapisano obiektów: 762
Pobieram fragment 0,1...
Zapisano obiektów: 3004
Pobieram fragment 0,2...
Zapisano obiektów: 612
Pobieram fragment 1,0...
Zapisano obiektów: 4174
Pobieram fragment 1,1...
Zapisano obiektów: 7677
Pobieram fragment 1,2...


ResponseStatusCodeError: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and the request queue is full.

## What to do

Retry your request using exponential backoff: wait a short interval, then retry.
If the request fails again, double the wait time and retry again.

## For heavy or automated use

This public endpoint is intended for browser-based and human-operated tools.
For automated or high-volume use, run a local Overpass API instance:

https://github.com/b1tw153/Overpass-API/blob/main/README.md


In [30]:

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import time
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
OSM_DIR = BASE / "osm"
OSM_DIR.mkdir(parents=True, exist_ok=True)

ox.settings.use_cache = True
ox.settings.cache_folder = str(BASE / "osm_cache")
ox.settings.requests_timeout = 180
ox.settings.overpass_rate_limit = True

df = pd.read_csv(
    BASE / "warszawa_mieszkania_dzielnice.csv"
)

south = df["latitude"].min() - 0.014
north = df["latitude"].max() + 0.014
west = df["longitude"].min() - 0.023
east = df["longitude"].max() + 0.023

lat_edges = np.linspace(south, north, 4)
lon_edges = np.linspace(west, east, 4)

tags = {
    "shop": True,
    "leisure": "park"
}

failed_tiles = []

for i in range(3):
    for j in range(3):

        path = OSM_DIR / f"osm_tile_{i}_{j}.gpkg"

        if path.exists():
            print(f"Fragment {i},{j}: już zapisany")
            continue

        bbox = (
            float(lon_edges[j]),
            float(lat_edges[i]),
            float(lon_edges[j+1]),
            float(lat_edges[i+1])
        )

        success = False

        for attempt in range(1, 4):
            try:
                print(
                    f"Pobieram {i},{j} "
                    f"(próba {attempt}/3)"
                )

                poi = ox.features_from_bbox(
                    bbox,
                    tags=tags
                )

                part = gpd.GeoDataFrame(
                    {
                        "osm_key": [
                            f"{kind}/{osmid}"
                            for kind, osmid in poi.index
                        ],
                        "shop": (
                            poi["shop"].fillna("").astype(str).tolist()
                            if "shop" in poi.columns
                            else [""] * len(poi)
                        ),
                        "leisure": (
                            poi["leisure"].fillna("").astype(str).tolist()
                            if "leisure" in poi.columns
                            else [""] * len(poi)
                        )
                    },
                    geometry=poi.geometry.to_list(),
                    crs=poi.crs
                )

                part.to_file(
                    path,
                    layer="pois",
                    driver="GPKG"
                )

                print(
                    f"Zapisano {i},{j}: {len(part)} obiektów"
                )

                success = True
                break

            except Exception as e:
                print(
                    f"Błąd: {type(e).__name__}: "
                    f"{str(e)[:180]}"
                )

                if attempt < 3:
                    delay = 20 * (2 ** (attempt - 1))
                    print(f"Ponowna próba za {delay} s")
                    time.sleep(delay)

        if not success:
            failed_tiles.append((i, j))
            print(f"Nie udało się pobrać {i},{j}")

        time.sleep(10)

saved = sorted(OSM_DIR.glob("osm_tile_*.gpkg"))

print("\nPODSUMOWANIE")
print("Zapisane fragmenty:", len(saved), "/9")
print("Nieudane fragmenty:", failed_tiles)

if len(saved) == 9:
    print("Wszystkie fragmenty zapisane!")
else:
    print("Możesz uruchomić kod ponownie.")


Fragment 0,0: już zapisany
Fragment 0,1: już zapisany
Fragment 0,2: już zapisany
Fragment 1,0: już zapisany
Fragment 1,1: już zapisany
Pobieram 1,2 (próba 1/3)
Błąd: ResponseStatusCodeError: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and the request queue is full.

##
Ponowna próba za 20 s
Pobieram 1,2 (próba 2/3)
Zapisano 1,2: 578 obiektów
Pobieram 2,0 (próba 1/3)
Zapisano 2,0: 1161 obiektów
Pobieram 2,1 (próba 1/3)
Zapisano 2,1: 1264 obiektów
Pobieram 2,2 (próba 1/3)
Zapisano 2,2: 584 obiektów

PODSUMOWANIE
Zapisane fragmenty: 9 /9
Nieudane fragmenty: []
Wszystkie fragmenty zapisane!


In [31]:

import geopandas as gpd
import pandas as pd
import numpy as np
import shapely
from shapely.strtree import STRtree
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
OSM_DIR = BASE / "osm"

# 1. Wczytanie 9 fragmentow
files = sorted(OSM_DIR.glob("osm_tile_*.gpkg"))

if len(files) != 9:
    raise ValueError(f"Brakuje fragmentow OSM: {len(files)}/9")

parts = [gpd.read_file(f) for f in files]

osm = gpd.GeoDataFrame(
    pd.concat(parts, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
)

print("Wszystkie obiekty:", len(osm))

# 2. Usuwanie duplikatow
osm = osm.drop_duplicates(subset="osm_key").copy()

osm = osm[
    osm.geometry.notna()
    & ~osm.geometry.is_empty
].copy()

print("Unikalne obiekty OSM:", len(osm))

# 3. Oddzielamy sklepy i parki
osm["shop"] = osm["shop"].fillna("").astype(str)
osm["leisure"] = osm["leisure"].fillna("").astype(str)

shops = osm[
    osm["shop"].str.strip().ne("")
    & ~osm["shop"].isin(["no", "vacant"])
].copy()

parks = osm[
    osm["leisure"].eq("park")
].copy()

print("Liczba sklepów:", len(shops))
print("Liczba parków:", len(parks))

# 4. Konwersja do ukladu metrycznego
shops = shops.to_crs("EPSG:2180")
parks = parks.to_crs("EPSG:2180")

# 5. Wczytanie mieszkan z dzielnicami
df = pd.read_csv(
    BASE / "warszawa_mieszkania_dzielnice.csv",
    dtype={"gid": "string"}
)

apartments = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(
        df["longitude"],
        df["latitude"]
    ),
    crs="EPSG:4326"
).to_crs("EPSG:2180")

# 6. Unikalne lokalizacje mieszkan
coords = np.column_stack([
    apartments.geometry.x.to_numpy(),
    apartments.geometry.y.to_numpy()
])

unique_coords, inverse = np.unique(
    np.round(coords, 3),
    axis=0,
    return_inverse=True
)

points = shapely.points(
    unique_coords[:, 0],
    unique_coords[:, 1]
)

print("Unikalne lokalizacje mieszkań:", len(points))

# 7. Funkcja liczaca obiekty w promieniu
def count_nearby(points, objects, radius):
    if objects.empty:
        raise ValueError("Brak obiektow do analizy")

    tree = STRtree(objects.geometry.to_numpy())
    counts = np.zeros(len(points), dtype=np.int32)

    for start in range(0, len(points), 500):
        end = min(start + 500, len(points))

        matches = tree.query(
            points[start:end],
            predicate="dwithin",
            distance=radius
        )

        counts[start:end] = np.bincount(
            matches[0],
            minlength=end - start
        )

    return counts

# 8. Liczenie sklepów
print("\nLiczenie sklepów do 500 m...")

shop_counts = count_nearby(
    points, shops, 500
)

# 9. Liczenie parków
print("Liczenie parków do 1000 m...")

park_counts = count_nearby(
    points, parks, 1000
)

# 10. Przypisanie wyniku mieszkaniom
df["shops_500m"] = shop_counts[inverse]
df["parks_1000m"] = park_counts[inverse]

# 11. Zapis
output = BASE / "warszawa_mieszkania_osm.csv"

df.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\nPODSUMOWANIE")
print("Liczba mieszkań:", len(df))
print("Średnia liczba sklepów:",
      round(df["shops_500m"].mean(), 2))
print("Średnia liczba parków:",
      round(df["parks_1000m"].mean(), 2))

print("\nSTATYSTYKI:")
display(
    df[["shops_500m", "parks_1000m"]].describe()
)

print("\nPRZYKŁADOWE MIESZKANIA:")
display(df[[
    "LOK_ADRES",
    "district",
    "distance_center_km",
    "shops_500m",
    "parks_1000m"
]].head(15))

print("\nZapisano:", output)


Wszystkie obiekty: 19816
Unikalne obiekty OSM: 19797
Liczba sklepów: 18710
Liczba parków: 474
Unikalne lokalizacje mieszkań: 15344

Liczenie sklepów do 500 m...
Liczenie parków do 1000 m...

PODSUMOWANIE
Liczba mieszkań: 89738
Średnia liczba sklepów: 64.13
Średnia liczba parków: 5.57

STATYSTYKI:


,shops_500m,parks_1000m
count,89738.000000,89738.000000
mean,64.126412,5.569268
std,65.233956,4.950069
min,0.000000,0.000000
25%,22.000000,2.000000
50%,45.000000,5.000000
75%,86.000000,8.000000
max,510.000000,30.000000



PRZYKŁADOWE MIESZKANIA:


,LOK_ADRES,district,distance_center_km,shops_500m,parks_1000m
0,MSC:Warszawa;UL:ulica Bokserska;NR_PORZ:73,Ursynów,6.861157,11,1
1,MSC:Warszawa;UL:ulica Budowlana;NR_PORZ:5,Targówek,5.577214,14,0
2,MSC:Warszawa;UL:ulica Franciszka Adolfa Achera...,Ursus,8.833854,119,13
3,MSC:Warszawa;UL:ulica Herbu Oksza;NR_PORZ:3,Ursus,8.895669,115,16
4,MSC:Warszawa;UL:ulica Dzieci Warszawy;NR_PORZ:17D,Ursus,8.150563,22,0
5,MSC:Warszawa;UL:ulica Herbu Oksza;NR_PORZ:3,Ursus,8.895669,115,16
6,MSC:Warszawa;UL:ulica Ogrodowa;NR_PORZ:5,Wola,1.247603,252,22
7,MSC:Warszawa;UL:ulica Burakowska;NR_PORZ:16B,Wola,3.267078,179,6
8,MSC:Warszawa;UL:ulica Waliców;NR_PORZ:11,Wola,1.028600,272,11
9,MSC:Warszawa;UL:ulica św. Stanisława;NR_PORZ:1A,Wola,3.397789,72,4



Zapisano: /content/drive/MyDrive/ZPA_RCN_Warszawa/warszawa_mieszkania_osm.csv


In [32]:

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import time
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
NEW_DIR = BASE / "osm_transport_schools"
NEW_DIR.mkdir(parents=True, exist_ok=True)

ox.settings.use_cache = True
ox.settings.cache_folder = str(BASE / "osm_cache")
ox.settings.requests_timeout = 180
ox.settings.overpass_rate_limit = True

df = pd.read_csv(
    BASE / "warszawa_mieszkania_osm.csv"
)

south = df["latitude"].min() - 0.014
north = df["latitude"].max() + 0.014
west = df["longitude"].min() - 0.023
east = df["longitude"].max() + 0.023

lat_edges = np.linspace(south, north, 4)
lon_edges = np.linspace(west, east, 4)

tags = {
    "amenity": "school",
    "highway": "bus_stop",
    "station": "subway",
    "subway": "yes"
}

failed = []

for i in range(3):
    for j in range(3):
        path = NEW_DIR / f"tile_{i}_{j}.gpkg"

        if path.exists():
            print(f"Fragment {i},{j}: już zapisany")
            continue

        bbox = (
            float(lon_edges[j]),
            float(lat_edges[i]),
            float(lon_edges[j+1]),
            float(lat_edges[i+1])
        )

        success = False

        for attempt in range(1, 4):
            try:
                print(f"Pobieram {i},{j} (próba {attempt})")

                poi = ox.features_from_bbox(
                    bbox,
                    tags=tags
                )

                def tag_values(name):
                    if name in poi.columns:
                        return poi[name].fillna("").astype(str).tolist()
                    return [""] * len(poi)

                part = gpd.GeoDataFrame(
                    {
                        "osm_key": [
                            f"{kind}/{osmid}"
                            for kind, osmid in poi.index
                        ],
                        "amenity": tag_values("amenity"),
                        "highway": tag_values("highway"),
                        "railway": tag_values("railway"),
                        "station": tag_values("station"),
                        "subway": tag_values("subway"),
                        "public_transport": tag_values(
                            "public_transport"
                        )
                    },
                    geometry=poi.geometry.to_list(),
                    crs=poi.crs
                )

                part.to_file(
                    path,
                    layer="pois",
                    driver="GPKG"
                )

                print(f"Zapisano: {len(part)} obiektów")
                success = True
                break

            except Exception as e:
                print(
                    f"Błąd: {type(e).__name__}: "
                    f"{str(e)[:150]}"
                )

                if attempt < 3:
                    delay = 20 * (2 ** (attempt-1))
                    time.sleep(delay)

        if not success:
            failed.append((i,j))

        time.sleep(10)

print("\nPODSUMOWANIE")
print("Zapisane fragmenty:",
      len(list(NEW_DIR.glob("tile_*.gpkg"))), "/9")
print("Nieudane:", failed)


Pobieram 0,0 (próba 1)
Zapisano: 418 obiektów
Pobieram 0,1 (próba 1)
Zapisano: 830 obiektów
Pobieram 0,2 (próba 1)
Zapisano: 415 obiektów
Pobieram 1,0 (próba 1)
Błąd: ResponseStatusCodeError: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and 
Pobieram 1,0 (próba 2)
Zapisano: 1177 obiektów
Pobieram 1,1 (próba 1)
Błąd: ResponseStatusCodeError: 'overpass-api.de' responded: 502 Bad Gateway # 502 Gateway Timeout

This Overpass API server is currently at capacity. All query workers are busy
and 
Pobieram 1,1 (próba 2)
Zapisano: 1709 obiektów
Pobieram 1,2 (próba 1)
Zapisano: 468 obiektów
Pobieram 2,0 (próba 1)
Zapisano: 491 obiektów
Pobieram 2,1 (próba 1)
Zapisano: 626 obiektów
Pobieram 2,2 (próba 1)
Zapisano: 346 obiektów

PODSUMOWANIE
Zapisane fragmenty: 9 /9
Nieudane: []


In [33]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
NEW_DIR = BASE / "osm_transport_schools"

files = sorted(NEW_DIR.glob("tile_*.gpkg"))

parts = [gpd.read_file(f) for f in files]

pois = gpd.GeoDataFrame(
    pd.concat(parts, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
)

pois = pois.drop_duplicates("osm_key").copy()

for col in ["amenity", "highway", "railway",
            "station", "subway", "public_transport"]:
    pois[col] = pois[col].fillna("").astype(str).str.lower()

schools = pois[pois["amenity"] == "school"]

bus_stops = pois[pois["highway"] == "bus_stop"]

metro = pois[
    (pois["station"] == "subway")
    | (
        (pois["subway"] == "yes")
        & (
            (pois["railway"] == "station")
            | (pois["public_transport"] == "station")
        )
    )
]

print("Unikalne obiekty OSM:", len(pois))
print("Szkoły:", len(schools))
print("Przystanki autobusowe:", len(bus_stops))
print("Obiekty metra:", len(metro))

print("\nTAGI METRA:")
display(
    metro[["railway", "station", "subway",
           "public_transport"]].value_counts().head(15)
)

Unikalne obiekty OSM: 6471
Szkoły: 979
Przystanki autobusowe: 5305
Obiekty metra: 62

TAGI METRA:


railway   station  subway  public_transport
station   subway   yes     station             38
proposed  subway   yes                         23
                                                1
Name: count, dtype: int64

In [34]:

import pandas as pd
import geopandas as gpd
import numpy as np
import shapely
from shapely.strtree import STRtree
from pathlib import Path

BASE = Path("/content/drive/MyDrive/ZPA_RCN_Warszawa")
NEW_DIR = BASE / "osm_transport_schools"

# 1. Wczytanie wszystkich obiektow OSM
files = sorted(NEW_DIR.glob("tile_*.gpkg"))

if len(files) != 9:
    raise ValueError("Brakuje fragmentow OSM!")

parts = [gpd.read_file(f) for f in files]

pois = gpd.GeoDataFrame(
    pd.concat(parts, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
)

pois = pois.drop_duplicates("osm_key").copy()

pois = pois[
    pois.geometry.notna()
    & ~pois.geometry.is_empty
].copy()

for col in [
    "amenity", "highway", "railway",
    "station", "subway", "public_transport"
]:
    pois[col] = (
        pois[col].fillna("")
        .astype(str).str.lower().str.strip()
    )

# 2. Wyodrebnienie kategorii
schools = pois[
    pois["amenity"].eq("school")
].copy()

bus_stops = pois[
    pois["highway"].eq("bus_stop")
].copy()

# TYLKO STACJE METRA - bez projektowanych
metro = pois[
    pois["railway"].eq("station")
    & pois["station"].eq("subway")
].copy()

print("Szkoly:", len(schools))
print("Przystanki autobusowe:", len(bus_stops))
print("Stacje metra:", len(metro))

if schools.empty or bus_stops.empty or metro.empty:
    raise ValueError("Brak obiektow w jednej z kategorii")

# 3. Konwersja do ukladu metrycznego
schools = schools.to_crs(2180)
bus_stops = bus_stops.to_crs(2180)
metro = metro.to_crs(2180)

# 4. Wczytanie mieszkan
df = pd.read_csv(
    BASE / "warszawa_mieszkania_osm.csv",
    dtype={"gid": "string"}
)

apartments = gpd.GeoDataFrame(
    df[["gid"]].copy(),
    geometry=gpd.points_from_xy(
        df["longitude"], df["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(2180)

# 5. Unikalne lokalizacje
coords = np.column_stack([
    apartments.geometry.x.to_numpy(),
    apartments.geometry.y.to_numpy()
])

unique_coords, inverse = np.unique(
    np.round(coords, 3),
    axis=0,
    return_inverse=True
)

points = shapely.points(
    unique_coords[:, 0],
    unique_coords[:, 1]
)

print("Unikalne lokalizacje:", len(points))

# 6. Liczenie obiektow w promieniu
def count_nearby(points, objects, radius):
    tree = STRtree(objects.geometry.to_numpy())
    counts = np.zeros(len(points), dtype=np.int32)

    for start in range(0, len(points), 500):
        end = min(start + 500, len(points))

        pairs = tree.query(
            points[start:end],
            predicate="dwithin",
            distance=radius
        )

        counts[start:end] = np.bincount(
            pairs[0],
            minlength=end-start
        )

    return counts

print("Liczenie szkol do 1000 m...")
school_counts = count_nearby(points, schools, 1000)

print("Liczenie przystankow do 500 m...")
stop_counts = count_nearby(points, bus_stops, 500)

# 7. Odleglosc do najblizszej stacji metra
print("Liczenie odleglosci do metra...")

metro_tree = STRtree(metro.geometry.to_numpy())

indices, distances = metro_tree.query_nearest(
    points,
    return_distance=True,
    all_matches=False
)

metro_distances = np.full(len(points), np.nan)
metro_distances[indices[0]] = distances

# 8. Dolaczenie cech
df["schools_1000m"] = school_counts[inverse]
df["bus_stops_500m"] = stop_counts[inverse]
df["distance_metro_m"] = metro_distances[inverse]

# 9. Kontrola i zapis
assert len(df) == 89738
assert df["gid"].is_unique
assert df[
    ["schools_1000m", "bus_stops_500m", "distance_metro_m"]
].notna().all().all()

output = BASE / "warszawa_mieszkania_full.csv"

df.to_csv(
    output,
    index=False,
    encoding="utf-8-sig"
)

print("\nPODSUMOWANIE")
print("Liczba mieszkan:", len(df))
print("Liczba kolumn:", len(df.columns))

display(df[[
    "shops_500m",
    "parks_1000m",
    "schools_1000m",
    "bus_stops_500m",
    "distance_metro_m"
]].describe().round(2))

print("\nPRZYKLADOWE MIESZKANIA:")
display(df[[
    "district",
    "shops_500m",
    "parks_1000m",
    "schools_1000m",
    "bus_stops_500m",
    "distance_metro_m"
]].head(10))

print("\nZapisano:", output)


Szkoly: 979
Przystanki autobusowe: 5305
Stacje metra: 38
Unikalne lokalizacje: 15344
Liczenie szkol do 1000 m...
Liczenie przystankow do 500 m...
Liczenie odleglosci do metra...

PODSUMOWANIE
Liczba mieszkan: 89738
Liczba kolumn: 45


,shops_500m,parks_1000m,schools_1000m,bus_stops_500m,distance_metro_m
count,89738.00,89738.00,89738.00,89738.00,89738.00
mean,64.13,5.57,10.34,10.57,2679.76
std,65.23,4.95,8.89,5.30,2230.94
min,0.00,0.00,0.00,0.00,28.81
25%,22.00,2.00,3.00,7.00,900.34
50%,45.00,5.00,7.00,10.00,2077.13
75%,86.00,8.00,15.00,14.00,3972.45
max,510.00,30.00,51.00,40.00,14630.98



PRZYKLADOWE MIESZKANIA:


,district,shops_500m,parks_1000m,schools_1000m,bus_stops_500m,distance_metro_m
0,Ursynów,11,1,2,3,2526.447786
1,Targówek,14,0,4,5,1453.440399
2,Ursus,119,13,11,16,4986.042367
3,Ursus,115,16,12,11,4816.573195
4,Ursus,22,0,6,11,5517.719217
5,Ursus,115,16,12,11,4816.573195
6,Wola,252,22,31,13,788.149838
7,Wola,179,6,42,13,973.752669
8,Wola,272,11,26,19,426.427088
9,Wola,72,4,15,17,1050.522942



Zapisano: /content/drive/MyDrive/ZPA_RCN_Warszawa/warszawa_mieszkania_full.csv
